# Short Video Generator for YouTube Shorts & Facebook ReelsGenerate short vertical videos for social media!## Quick Start1. Enable GPU (Runtime -> Change runtime type -> GPU)2. Run all cells (Runtime -> Run all)3. Enter your prompt4. Click Generate Short Video5. Download your video!

In [ ]:
!pip install -q torch torchvision transformers diffusers accelerate gradio imageio-ffmpegprint("Packages installed!")

In [ ]:
import torchimport numpy as npimport gradio as grimport osfrom datetime import datetimefrom PIL import Imageimport imageioimport warningswarnings.filterwarnings("ignore")device = "cuda" if torch.cuda.is_available() else "cpu"print(f"Device: {device}")if device == "cuda":    print(f"GPU: {torch.cuda.get_device_name(0)}")    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3    print(f"VRAM: {vram:.1f} GB")

In [ ]:
print("Loading model...")from diffusers import DiffusionPipeline, DPMSolverMultistepSchedulerpipe = DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16").to(device)pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)pipe.enable_model_cpu_offload()print("Model loaded!")

In [ ]:
def generate_short_video(prompt, negative_prompt="", num_frames=8, num_inference_steps=15, guidance_scale=7.5, fps=8):    print(f"Generating: {prompt[:50]}...")    if device == "cuda": torch.cuda.empty_cache()    with torch.no_grad():        video_frames = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).frames[0]    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")    output_path = f"/content/short_video_{timestamp}.mp4"    writer = imageio.get_writer(output_path, fps=fps)    for frame in video_frames:        if isinstance(frame, Image.Image): frame = np.array(frame)        writer.append_data(frame)    writer.close()    print(f"Done! Duration: {num_frames/fps:.1f}s, Size: {os.path.getsize(output_path)/(1024*1024):.1f}MB")    return output_path

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:    gr.Markdown("# Short Video Generator\nGenerate short videos for YouTube Shorts & Facebook Reels!")    with gr.Row():        with gr.Column():            prompt_input = gr.Textbox(label="Prompt", placeholder="Describe your video...", value="A sunset over the ocean")            negative_prompt = gr.Textbox(label="Negative", value="blurry, low quality")            with gr.Accordion("Settings", open=False):                num_frames = gr.Slider(6, 12, value=8, label="Frames")                num_steps = gr.Slider(10, 20, value=15, label="Steps")                guidance_scale = gr.Slider(5.0, 10.0, value=7.5, label="Guidance")                fps = gr.Slider(6, 12, value=8, label="FPS")            generate_btn = gr.Button("Generate Short Video", variant="primary", size="lg")        with gr.Column():            video_output = gr.Video(label="Your Video", autoplay=True)            status = gr.Textbox(label="Status", value="Ready!", interactive=False)    gr.Examples([["A sunset over the ocean", "blurry"], ["A cat playing", "low quality"]], inputs=[prompt_input, negative_prompt])    def on_generate(prompt, neg, frames, steps, guidance, fps_val):        if not prompt.strip(): return None, "Enter a prompt!"        yield None, "Generating... Wait 1-2 min"        try:            video_path = generate_short_video(prompt, neg, int(frames), int(steps), float(guidance), int(fps_val))            return video_path, f"Done! {os.path.basename(video_path)}"        except Exception as e:            return None, f"Error: {str(e)}"    generate_btn.click(fn=on_generate, inputs=[prompt_input, negative_prompt, num_frames, num_steps, guidance_scale, fps], outputs=[video_output, status])print("UI ready!")

In [ ]:
print("Launching Short Video Generator...")demo.launch(share=True, debug=False, show_error=True)